# Scaling Inference-Time Compute for Mathematical Reasoning

## AIMO Progress Prize 3 - Comprehensive Technical Writeup

**Author:** Pawan Mali  
**Competition:** [AI Mathematical Olympiad Progress Prize 3](https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-3)  
**Best Score:** 42/50 (V40)  
**Experiments:** 157+ versions documented

---

### Table of Contents

1. [Competition Overview](#1-competition-overview)
2. [Key Results Summary](#2-key-results-summary)
3. [Model Selection & Infrastructure](#3-model-selection--infrastructure)
4. [Entropy-Weighted Voting System](#4-entropy-weighted-voting-system)
5. [Prompt Engineering Findings](#5-prompt-engineering-findings)
6. [Ablation Studies](#6-ablation-studies)
7. [Failed Experiments & Learnings](#7-failed-experiments--learnings)
8. [Interactive Demonstrations](#8-interactive-demonstrations)
9. [Best Practices & Recommendations](#9-best-practices--recommendations)
10. [Conclusion](#10-conclusion)

---

## 1. Competition Overview

### The Challenge

The AI Mathematical Olympiad (AIMO) Progress Prize 3 represents one of the most challenging benchmarks for mathematical reasoning in artificial intelligence:

| Aspect | Details |
|--------|----------|
| **Problems** | 50 hidden IMO-level mathematical problems |
| **Time Limit** | 4.8 hours total runtime |
| **Hardware** | NVIDIA H100 80GB GPU |
| **Answer Format** | Non-negative integers between 0 and 99,999 |
| **Prize Pool** | $2.2 million for achieving 47+ correct answers |
| **Current Best** | 46/50 (competition winner) |

### Why This Competition Matters

Mathematical reasoning requires:
- **Multi-step logical deduction** - Problems cannot be solved in a single inference
- **Precise calculations** - Small arithmetic errors cascade to wrong answers
- **Creative problem-solving** - Standard patterns often don't apply
- **Self-verification** - Models must check their own work

This makes AIMO3 an ideal testbed for inference-time scaling strategies.

## 2. Key Results Summary

### Performance Timeline

| Version | Date | Score | Configuration | Key Change |
|---------|------|-------|---------------|------------|
| V1 | Jan 15 | 12/50 | Baseline | Initial setup |
| V15 | Jan 25 | 28/50 | + TIR | Tool-integrated reasoning |
| V25 | Feb 1 | 35/50 | + Voting | Simple majority voting |
| V35 | Feb 4 | 38/50 | + Entropy | Mean entropy weighting |
| **V40** | **Feb 6** | **42/50** | **+ Position weighting** | **Best result** |
| V125 | Mar 19 | 41/50 | Stable baseline | Reproducible config |
| V127 | Mar 22 | 33/50 | Verbose prompts | -9 regression |
| V142 | Apr 5 | 30/50 | Bayesian voting | -11 regression |

### Critical Findings

| Finding | Impact | Evidence |
|---------|--------|----------|
| Simple 3-line prompts beat verbose instructions | **+9 points** | V40 (42) vs V127 (33) |
| 5-component weighted entropy beats mean entropy | **+6 points** | V40 (42) vs V136 (36) |
| Temperature 1.0 beats 0.5 with voting | **+8 points** | V40 (42) vs V41 (34) |
| 65K context beats 131K context | **+6 points** | V125 (41) vs V135 (35) |
| 8 attempts with early-stop at 4 | **+4 points** | Optimal balance |

In [ ]:
# Setup - Install required packages
!pip install -q numpy matplotlib seaborn pandas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import defaultdict
from math import log, sqrt, exp
from typing import List, Dict, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("Environment ready!")

## 3. Model Selection & Infrastructure

### Model Comparison

We evaluated multiple models before selecting GPT-OSS-120B:

| Model | Parameters | Context | Score Range | Notes |
|-------|------------|---------|-------------|-------|
| DeepSeek-R1-32B | 32B | 64K | 28-32 | Good reasoning, limited capacity |
| Qwen2.5-Math-72B | 72B | 32K | 30-35 | Math-specialized, short context |
| **GPT-OSS-120B** | **120B** | **131K** | **38-42** | **Best performance** |

### Why GPT-OSS-120B?

1. **Parameter count**: 120B parameters provide strong reasoning capabilities
2. **Context window**: 131K tokens allow for extended chain-of-thought
3. **FP8 KV cache**: Enables efficient memory utilization on H100
4. **vLLM compatibility**: Optimized for high-throughput inference

### Model Links

- **Kaggle Models:** https://www.kaggle.com/models/danielhanchen/gpt-oss-120b
- **HuggingFace:** https://huggingface.co/unsloth/gpt-oss-120B

In [ ]:
# Visualize model comparison
models = ['DeepSeek-R1\n32B', 'Qwen2.5-Math\n72B', 'GPT-OSS\n120B']
scores_min = [28, 30, 38]
scores_max = [32, 35, 42]
scores_avg = [(a+b)/2 for a, b in zip(scores_min, scores_max)]

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(models))
width = 0.6

bars = ax.bar(x, scores_avg, width, color=['#3498db', '#2ecc71', '#e74c3c'], alpha=0.8)
ax.errorbar(x, scores_avg, yerr=[np.array(scores_avg)-np.array(scores_min), 
                                  np.array(scores_max)-np.array(scores_avg)], 
            fmt='none', color='black', capsize=5, capthick=2)

ax.set_ylabel('Score (out of 50)', fontsize=12)
ax.set_title('Model Performance Comparison on AIMO3', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylim(0, 50)
ax.axhline(y=47, color='gold', linestyle='--', linewidth=2, label='Prize threshold (47)')
ax.legend()

# Add value labels
for i, (bar, smin, smax) in enumerate(zip(bars, scores_min, scores_max)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
            f'{smin}-{smax}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

### vLLM Server Configuration

Optimal configuration for H100 80GB:

In [ ]:
# vLLM Server Configuration (Best Settings)
VLLM_CONFIG = {
    'model': '/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1',
    'served_model_name': 'gpt-oss',
    'tensor_parallel_size': 1,
    'max_num_seqs': 256,
    'gpu_memory_utilization': 0.96,
    'dtype': 'auto',
    'kv_cache_dtype': 'fp8_e4m3',  # Critical for memory efficiency
    'max_model_len': 65536,         # Not 131K - see ablation
    'enable_prefix_caching': True,
    'async_scheduling': True,
}

print("vLLM Server Configuration:")
print("=" * 50)
for key, value in VLLM_CONFIG.items():
    print(f"{key:25} : {value}")

### Infrastructure Architecture

```
                    AIMO3 Inference Pipeline
                    ========================

    ┌─────────────────────────────────────────────────────────┐
    │                    Problem Input                         │
    │              (IMO-level math problem)                    │
    └─────────────────────────┬───────────────────────────────┘
                              │
                              ▼
    ┌─────────────────────────────────────────────────────────┐
    │                   vLLM Server                            │
    │  ┌─────────────────────────────────────────────────┐    │
    │  │           GPT-OSS-120B (120B params)            │    │
    │  │  • FP8 KV Cache (memory efficient)              │    │
    │  │  • 65K context window                           │    │
    │  │  • Prefix caching enabled                       │    │
    │  └─────────────────────────────────────────────────┘    │
    └─────────────────────────┬───────────────────────────────┘
                              │
            ┌─────────────────┼─────────────────┐
            │                 │                 │
            ▼                 ▼                 ▼
    ┌───────────────┐ ┌───────────────┐ ┌───────────────┐
    │   Attempt 1   │ │   Attempt 2   │ │  ... (8 total)│
    │ ┌───────────┐ │ │ ┌───────────┐ │ │ ┌───────────┐ │
    │ │  Reason   │ │ │ │  Reason   │ │ │ │  Reason   │ │
    │ │     +     │ │ │ │     +     │ │ │ │     +     │ │
    │ │   Code    │ │ │ │   Code    │ │ │ │   Code    │ │
    │ └───────────┘ │ │ └───────────┘ │ │ └───────────┘ │
    │ Answer: 42    │ │ Answer: 42    │ │ Answer: 37    │
    │ Entropy: 0.3  │ │ Entropy: 0.4  │ │ Entropy: 1.8  │
    └───────────────┘ └───────────────┘ └───────────────┘
            │                 │                 │
            └─────────────────┼─────────────────┘
                              │
                              ▼
    ┌─────────────────────────────────────────────────────────┐
    │              Entropy-Weighted Voting                     │
    │  ┌─────────────────────────────────────────────────┐    │
    │  │  5-Component Weighted Entropy Calculation:      │    │
    │  │  • Mean entropy (30%)                           │    │
    │  │  • Position-weighted entropy (40%)              │    │
    │  │  • Variance penalty (20%)                       │    │
    │  │  • High-entropy penalty                         │    │
    │  │  • Low-entropy streak bonus                     │    │
    │  └─────────────────────────────────────────────────┘    │
    │                                                          │
    │  Vote weights: w = 1 / max(entropy, 1e-9)               │
    │  Answer 42: weight = 3.33 + 2.50 = 5.83                 │
    │  Answer 37: weight = 0.56                               │
    └─────────────────────────┬───────────────────────────────┘
                              │
                              ▼
    ┌─────────────────────────────────────────────────────────┐
    │                  Final Answer: 42                        │
    │           (Highest confidence-weighted votes)            │
    └─────────────────────────────────────────────────────────┘
```

## 4. Entropy-Weighted Voting System

### The Core Innovation

Our key contribution is a **5-component weighted entropy** system that significantly outperforms simple majority voting (+6 points).

### Why Entropy Matters

**Shannon Entropy** measures uncertainty in a probability distribution:

$$H(X) = -\sum_{i} p_i \log(p_i)$$

- **Low entropy** → Model is confident (probability concentrated on one token)
- **High entropy** → Model is uncertain (probability spread across many tokens)

### The 5 Components

| Component | Weight | Purpose |
|-----------|--------|----------|
| Mean Entropy | 30% | Overall confidence across all tokens |
| Position-Weighted Entropy | 40% | Emphasize confidence at final answer |
| Variance Penalty | 20% | Penalize inconsistent confidence |
| High-Entropy Penalty | - | Penalize stretches of uncertainty |
| Low-Entropy Streak Bonus | - | Reward consistent confidence |

In [ ]:
def compute_token_entropy(probs: List[float]) -> float:
    """
    Compute Shannon entropy for a single token's probability distribution.
    
    Args:
        probs: List of probabilities for each possible token
        
    Returns:
        Entropy value (higher = more uncertain)
        
    Example:
        [0.99, 0.01] → entropy ≈ 0.08 (very confident)
        [0.5, 0.5]   → entropy ≈ 0.69 (uncertain)
        [0.25, 0.25, 0.25, 0.25] → entropy ≈ 1.39 (very uncertain)
    """
    entropy = 0.0
    for p in probs:
        if p > 1e-10:  # Avoid log(0)
            entropy -= p * log(p)
    return entropy

# Demonstrate entropy calculation
examples = [
    ([0.99, 0.01], "Very confident"),
    ([0.9, 0.1], "Confident"),
    ([0.7, 0.3], "Somewhat confident"),
    ([0.5, 0.5], "Uncertain (binary)"),
    ([0.25, 0.25, 0.25, 0.25], "Very uncertain (4-way)"),
    ([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1], "Maximum uncertainty (10-way)"),
]

print("Entropy Examples:")
print("=" * 60)
for probs, description in examples:
    ent = compute_token_entropy(probs)
    print(f"{description:35} → entropy = {ent:.4f}")

In [ ]:
def compute_weighted_entropy(
    logprobs: List[List[float]],
    decay: float = 0.995,
    mean_weight: float = 0.3,
    position_weight: float = 0.4,
    variance_weight: float = 0.2,
    high_entropy_threshold: float = 2.0,
    high_entropy_penalty: float = 0.3
) -> float:
    """
    Compute 5-component weighted entropy for a solution.
    
    This is the core innovation that provides +6 points over simple averaging.
    
    Components:
    1. Mean entropy (30%): Average entropy across all tokens
    2. Position-weighted entropy (40%): Exponential decay emphasizing final tokens
    3. Variance penalty (20%): sqrt(variance) penalizes inconsistent confidence
    4. High-entropy penalty: Ratio of high-entropy tokens × penalty weight
    5. (Implicit) Low-entropy streaks contribute to lower position-weighted entropy
    
    Args:
        logprobs: List of probability distributions, one per token
        decay: Exponential decay factor for position weighting (0.995 works best)
        mean_weight: Weight for mean entropy component
        position_weight: Weight for position-weighted entropy
        variance_weight: Weight for variance penalty
        high_entropy_threshold: Threshold for "high entropy" classification
        high_entropy_penalty: Penalty weight for high-entropy tokens
    
    Returns:
        Combined weighted entropy score (lower = more confident)
    """
    if not logprobs:
        return float('inf')
    
    # Calculate per-token entropy
    entropies = [compute_token_entropy(probs) for probs in logprobs]
    n = len(entropies)
    
    if n == 0:
        return float('inf')
    
    # Component 1: Mean entropy (30%)
    mean_ent = sum(entropies) / n
    
    # Component 2: Position-weighted entropy (40%)
    # Key insight: Mathematical solutions have uncertain exploration early,
    # but should be confident at the final answer.
    # Exponential decay emphasizes tokens near the end.
    weights = [decay ** (n - i - 1) for i in range(n)]
    position_weighted_ent = sum(e * w for e, w in zip(entropies, weights)) / sum(weights)
    
    # Component 3: Variance penalty (20%)
    # Penalize solutions with wildly inconsistent confidence
    variance = sum((e - mean_ent)**2 for e in entropies) / n
    variance_penalty = sqrt(variance)
    
    # Component 4: High-entropy penalty
    # Count proportion of tokens with entropy > threshold
    high_ent_ratio = sum(1 for e in entropies if e > high_entropy_threshold) / n
    high_ent_penalty = high_ent_ratio * 3.0  # Scale factor
    
    # Combine all components
    weighted_entropy = (
        mean_weight * mean_ent +
        position_weight * position_weighted_ent +
        variance_weight * variance_penalty +
        high_entropy_penalty * high_ent_penalty
    )
    
    return weighted_entropy


def select_answer(results: List[Dict]) -> Tuple[Optional[int], float, Dict]:
    """
    Select the best answer using entropy-weighted voting.
    
    Args:
        results: List of solution attempts, each with 'answer' and 'logprobs'
    
    Returns:
        Tuple of (selected_answer, total_weight, all_weights_by_answer)
    """
    answer_weights = defaultdict(float)
    answer_details = defaultdict(list)
    
    for i, result in enumerate(results):
        answer = result.get('answer')
        if answer is None:
            continue
            
        logprobs = result.get('logprobs', [])
        entropy = compute_weighted_entropy(logprobs)
        
        # Vote weight is inverse of entropy
        # Low entropy → high confidence → high weight
        weight = 1.0 / max(entropy, 1e-9)
        
        answer_weights[answer] += weight
        answer_details[answer].append({
            'attempt': i + 1,
            'entropy': entropy,
            'weight': weight
        })
    
    if not answer_weights:
        return None, 0.0, {}
    
    best_answer = max(answer_weights.items(), key=lambda x: x[1])
    return best_answer[0], best_answer[1], dict(answer_details)

print("Entropy-weighted voting functions defined!")

In [ ]:
# Visualize position weighting
n_tokens = 100
decay = 0.995

positions = np.arange(n_tokens)
weights = np.array([decay ** (n_tokens - i - 1) for i in range(n_tokens)])

fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(positions, weights, alpha=0.3, color='blue')
ax.plot(positions, weights, 'b-', linewidth=2)

ax.axvline(x=n_tokens-10, color='red', linestyle='--', linewidth=2, 
           label='Final answer region')
ax.axvspan(n_tokens-10, n_tokens, alpha=0.2, color='red')

ax.set_xlabel('Token Position', fontsize=12)
ax.set_ylabel('Weight', fontsize=12)
ax.set_title('Position Weighting: Final Tokens Matter Most (decay=0.995)', 
             fontsize=14, fontweight='bold')
ax.legend(loc='upper left')

# Annotate
ax.annotate('Early exploration\n(low weight)', xy=(10, 0.65), fontsize=10,
            ha='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.annotate('Final answer\n(high weight)', xy=(95, 0.95), fontsize=10,
            ha='center', bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))

plt.tight_layout()
plt.show()

### Why Position Weighting Works

Mathematical reasoning has a characteristic **entropy profile**:

1. **Early tokens (problem understanding)**: High entropy as model considers multiple approaches
2. **Middle tokens (exploration)**: Variable entropy during calculation steps
3. **Final tokens (answer)**: Should be LOW entropy if solution is correct

Position weighting captures this structure:
- **Correct solutions**: Confident final answer → low position-weighted entropy → high vote weight
- **Incorrect solutions**: Uncertain final answer → high position-weighted entropy → low vote weight

In [ ]:
# Visualize entropy profiles of correct vs incorrect solutions
np.random.seed(42)
n_tokens = 100

# Simulated entropy profiles
# Correct solution: high early, low at end
correct_profile = np.concatenate([
    np.random.uniform(1.0, 2.0, 20),   # Understanding phase
    np.random.uniform(0.5, 1.5, 60),   # Calculation phase
    np.random.uniform(0.1, 0.3, 20),   # Confident answer
])

# Incorrect solution: uncertain throughout, especially at end
incorrect_profile = np.concatenate([
    np.random.uniform(1.2, 2.2, 20),   # Understanding phase
    np.random.uniform(1.0, 2.0, 60),   # Confused calculation
    np.random.uniform(1.5, 2.5, 20),   # Uncertain answer (guessing)
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot correct solution
axes[0].fill_between(range(n_tokens), correct_profile, alpha=0.3, color='green')
axes[0].plot(correct_profile, 'g-', linewidth=2, label='Token entropy')
axes[0].axhline(y=np.mean(correct_profile), color='blue', linestyle='--', 
                label=f'Mean entropy: {np.mean(correct_profile):.2f}')
axes[0].axvspan(80, 100, alpha=0.2, color='gold')
axes[0].set_title('CORRECT Solution: Confident Final Answer', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Token Position')
axes[0].set_ylabel('Entropy')
axes[0].legend()
axes[0].set_ylim(0, 3)

# Plot incorrect solution
axes[1].fill_between(range(n_tokens), incorrect_profile, alpha=0.3, color='red')
axes[1].plot(incorrect_profile, 'r-', linewidth=2, label='Token entropy')
axes[1].axhline(y=np.mean(incorrect_profile), color='blue', linestyle='--', 
                label=f'Mean entropy: {np.mean(incorrect_profile):.2f}')
axes[1].axvspan(80, 100, alpha=0.2, color='gold')
axes[1].set_title('INCORRECT Solution: Uncertain Final Answer', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Token Position')
axes[1].set_ylabel('Entropy')
axes[1].legend()
axes[1].set_ylim(0, 3)

plt.suptitle('Entropy Profiles: Why Position Weighting Works', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Calculate weighted entropy for both
correct_logprobs = [[0.99 - e*0.3, e*0.3 + 0.01] for e in correct_profile]
incorrect_logprobs = [[0.6 - e*0.1, 0.4 + e*0.1] for e in incorrect_profile]

correct_entropy = compute_weighted_entropy(correct_logprobs)
incorrect_entropy = compute_weighted_entropy(incorrect_logprobs)

print(f"\nWeighted Entropy Comparison:")
print(f"  Correct solution:   {correct_entropy:.4f} → Vote weight: {1/correct_entropy:.2f}")
print(f"  Incorrect solution: {incorrect_entropy:.4f} → Vote weight: {1/incorrect_entropy:.2f}")
print(f"  Correct solution gets {(1/correct_entropy)/(1/incorrect_entropy):.1f}x more voting power!")

## 5. Prompt Engineering Findings

### The Counterintuitive Discovery

**Simple prompts dramatically outperform complex prompts** (+9 points).

This was our most surprising finding and contradicts common prompt engineering wisdom.

In [ ]:
# Compare prompts
SIMPLE_PROMPT = """
You are a world-class IMO competitor.
The final answer must be 0-99999.
Place answer inside \\boxed{}.
"""

VERBOSE_PROMPT = """
You are an elite mathematical problem solver with expertise at the International 
Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through 
rigorous mathematical reasoning.

# Problem-Solving Approach:
1. UNDERSTAND: Carefully read and rephrase the problem in your own words. 
   Identify what is given, what needs to be found, and any constraints.
2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, 
   techniques, patterns, or analogous problems. Don't commit to one approach immediately.
3. PLAN: Select the most promising approach and outline key steps before executing.
4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.
5. VERIFY: Check your answer by substituting back, testing edge cases, or using 
   alternative methods. Ensure logical consistency throughout.

# Mathematical Reasoning Principles:
- Break complex problems into smaller, manageable sub-problems
- Look for patterns, symmetries, and special cases that provide insight
- Use concrete examples to build intuition before generalizing
- Consider extreme cases and boundary conditions
- If stuck, try working backwards from the desired result
- Be willing to restart with a different approach if needed

# Verification Requirements:
- Cross-check arithmetic and algebraic manipulations
- Verify that your solution satisfies all problem constraints
- Test your answer with simple cases or special values when possible
- Ensure dimensional consistency and reasonableness of the result

# Output Format:
The final answer must be a non-negative integer between 0 and 99999.
Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}

Think step-by-step and show your complete reasoning process. Quality of reasoning 
is as important as the final answer.
"""

print("SIMPLE PROMPT (V40 - 42/50):")
print("=" * 50)
print(SIMPLE_PROMPT)
print(f"\nLength: {len(SIMPLE_PROMPT)} characters, ~{len(SIMPLE_PROMPT.split())} words")
print()
print("VERBOSE PROMPT (V127 - 33/50):")
print("=" * 50)
print(VERBOSE_PROMPT[:500] + "...")
print(f"\nLength: {len(VERBOSE_PROMPT)} characters, ~{len(VERBOSE_PROMPT.split())} words")

In [ ]:
# Visualize prompt length vs performance
prompt_data = [
    ('Simple\n(3 lines)', 42, len(SIMPLE_PROMPT.split())),
    ('Moderate\n(~50 words)', 41, 50),
    ('Detailed\n(~100 words)', 38, 100),
    ('Verbose\n(~250 words)', 33, 250),
]

labels, scores, words = zip(*prompt_data)

fig, ax1 = plt.subplots(figsize=(10, 6))

x = np.arange(len(labels))
width = 0.35

# Score bars
bars1 = ax1.bar(x - width/2, scores, width, label='Score', color='#2ecc71', alpha=0.8)
ax1.set_ylabel('Score (out of 50)', color='#2ecc71', fontsize=12)
ax1.tick_params(axis='y', labelcolor='#2ecc71')
ax1.set_ylim(0, 50)

# Word count bars
ax2 = ax1.twinx()
bars2 = ax2.bar(x + width/2, words, width, label='Word Count', color='#e74c3c', alpha=0.8)
ax2.set_ylabel('Prompt Word Count', color='#e74c3c', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#e74c3c')

ax1.set_xlabel('Prompt Type', fontsize=12)
ax1.set_title('Prompt Complexity vs Performance\n(Simpler is Better!)', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=10)

# Add value labels
for bar, score in zip(bars1, scores):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'{score}', ha='center', va='bottom', fontsize=11, fontweight='bold', color='#2ecc71')

fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88))
plt.tight_layout()
plt.show()

### Why Simple Prompts Win

We hypothesize three contributing factors:

1. **Internalized Knowledge**
   - GPT-OSS-120B was trained on extensive mathematical content
   - The model already has effective problem-solving strategies
   - Elaborate prompts may override learned behaviors with inferior explicit instructions

2. **Context Conservation**
   - Verbose prompts consume context tokens (~500 tokens)
   - Those tokens could otherwise support longer reasoning chains
   - With 65K context, 500 tokens = 0.8% overhead, but compounded across attempts

3. **Conflicting Instructions**
   - Detailed multi-step instructions may introduce contradictions
   - "Verify your answer" + "Show all steps" + "Be concise" creates confusion
   - Simple prompts have no internal conflicts

## 6. Ablation Studies

### Temperature Study

In [ ]:
# Temperature ablation
temp_data = {
    'Temperature': [0.3, 0.5, 0.7, 1.0, 1.2],
    'Score': [32, 34, 38, 42, 39],
    'Diversity': ['Very Low', 'Low', 'Medium', 'High', 'Too High'],
}

df_temp = pd.DataFrame(temp_data)

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']
bars = ax.bar(df_temp['Temperature'].astype(str), df_temp['Score'], color=colors, alpha=0.8)

ax.set_xlabel('Temperature', fontsize=12)
ax.set_ylabel('Score (out of 50)', fontsize=12)
ax.set_title('Temperature Ablation: Why 1.0 is Optimal for Voting', fontsize=14, fontweight='bold')
ax.set_ylim(0, 50)

# Highlight optimal
bars[3].set_edgecolor('black')
bars[3].set_linewidth(3)

# Add annotations
for bar, (_, row) in zip(bars, df_temp.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{row['Score']}\n({row['Diversity']})", 
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("Key Insight: Higher temperature creates DIVERSE solution paths.")
print("With 8 parallel attempts + voting, diversity helps find correct answers.")
print("Temperature 0.5 causes all attempts to converge to same (often wrong) solution.")

In [ ]:
# Context length ablation
context_data = {
    'Context (K)': ['32K', '65K', '98K', '131K'],
    'Score': [35, 41, 38, 35],
    'Concurrency': [8.5, 7.1, 4.8, 3.4],
}

df_context = pd.DataFrame(context_data)

fig, ax1 = plt.subplots(figsize=(10, 6))

x = np.arange(len(df_context))
width = 0.35

bars1 = ax1.bar(x - width/2, df_context['Score'], width, label='Score', color='#2ecc71', alpha=0.8)
ax1.set_ylabel('Score (out of 50)', color='#2ecc71', fontsize=12)
ax1.set_ylim(0, 50)

ax2 = ax1.twinx()
bars2 = ax2.bar(x + width/2, df_context['Concurrency'], width, label='Concurrency', color='#3498db', alpha=0.8)
ax2.set_ylabel('Max Concurrent Requests', color='#3498db', fontsize=12)

# Highlight optimal
bars1[1].set_edgecolor('black')
bars1[1].set_linewidth(3)

ax1.set_xlabel('Context Window', fontsize=12)
ax1.set_title('Context Length vs Performance\n(Larger ≠ Better)', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(df_context['Context (K)'])

fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88))
plt.tight_layout()
plt.show()

print("Key Insight: Larger context REDUCES concurrency.")
print("With 131K context, only 3.4 parallel requests fit in memory.")
print("65K is the sweet spot: enough context for reasoning, high enough concurrency.")

In [ ]:
# Voting method comparison
voting_data = {
    'Method': ['Majority\nVote', 'Mean\nEntropy', 'Position\nWeighted', '5-Component\nWeighted', 'Bayesian\nVOI'],
    'Score': [35, 36, 40, 42, 30],
    'Description': [
        'Simple count',
        'w = 1/mean(H)',
        'w = 1/pos_weighted(H)',
        'Full system',
        'Complex VOI'
    ]
}

df_voting = pd.DataFrame(voting_data)

fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#95a5a6', '#3498db', '#2ecc71', '#27ae60', '#e74c3c']
bars = ax.barh(df_voting['Method'], df_voting['Score'], color=colors, alpha=0.8)

ax.set_xlabel('Score (out of 50)', fontsize=12)
ax.set_title('Voting Method Comparison', fontsize=14, fontweight='bold')
ax.set_xlim(0, 50)

# Highlight best
bars[3].set_edgecolor('black')
bars[3].set_linewidth(3)

# Add labels
for bar, desc in zip(bars, df_voting['Description']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f"{bar.get_width():.0f} ({desc})", va='center', fontsize=10)

# Add delta annotations
ax.annotate('', xy=(42, 3.7), xytext=(35, 0.3),
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.text(38.5, 2, '+7 pts', fontsize=12, color='green', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Failed Experiments & Learnings

### Comprehensive Failure Analysis

Understanding what **doesn't work** is as valuable as knowing what works.

In [ ]:
# Failed experiments data
failures = [
    {
        'experiment': 'Bayesian Voting with VOI',
        'version': 'V142',
        'baseline': 41,
        'result': 30,
        'delta': -11,
        'hypothesis': 'Complex aggregation would improve selection',
        'why_failed': 'Over-penalized uncertainty in ways uncorrelated with correctness'
    },
    {
        'experiment': 'Verbose Prompts',
        'version': 'V127',
        'baseline': 42,
        'result': 33,
        'delta': -9,
        'hypothesis': 'Detailed instructions would guide reasoning',
        'why_failed': 'Conflicted with model\'s learned strategies; wasted context'
    },
    {
        'experiment': '131K Context',
        'version': 'V135',
        'baseline': 41,
        'result': 35,
        'delta': -6,
        'hypothesis': 'More context enables longer reasoning',
        'why_failed': 'Reduced concurrency from 7.1x to 3.4x; fewer attempts completed'
    },
    {
        'experiment': 'Fine-tuned Model (mismatched)',
        'version': 'V138',
        'baseline': 41,
        'result': 35,
        'delta': -6,
        'hypothesis': 'Fine-tuned model would perform better',
        'why_failed': 'Used with different pipeline than training; broke optimizations'
    },
    {
        'experiment': 'Combined 6 Improvements',
        'version': 'V131',
        'baseline': 41,
        'result': 35,
        'delta': -6,
        'hypothesis': 'Stacking improvements would compound gains',
        'why_failed': 'Interactions between changes were unpredictable and negative'
    },
    {
        'experiment': 'Self-Refinement',
        'version': 'V134',
        'baseline': 41,
        'result': 37,
        'delta': -4,
        'hypothesis': 'Model reviewing own answers would catch errors',
        'why_failed': 'Model "corrected" correct answers to incorrect ones'
    },
    {
        'experiment': 'Answer Verification',
        'version': 'V133',
        'baseline': 41,
        'result': 41,
        'delta': 0,
        'hypothesis': 'Verification prompt would filter wrong answers',
        'why_failed': 'Model equally likely to verify wrong answers as correct'
    },
]

df_failures = pd.DataFrame(failures)

# Display as formatted table
print("FAILED EXPERIMENTS SUMMARY")
print("=" * 100)
for _, row in df_failures.iterrows():
    print(f"\n{row['experiment']} ({row['version']})")
    print(f"  Baseline: {row['baseline']} → Result: {row['result']} (Δ = {row['delta']:+d})")
    print(f"  Hypothesis: {row['hypothesis']}")
    print(f"  Why Failed: {row['why_failed']}")

In [ ]:
# Visualize failures
fig, ax = plt.subplots(figsize=(12, 6))

experiments = [f"{row['experiment']}\n({row['version']})" for _, row in df_failures.iterrows()]
deltas = df_failures['delta'].values

colors = ['#e74c3c' if d < 0 else '#f1c40f' if d == 0 else '#2ecc71' for d in deltas]
bars = ax.barh(experiments, deltas, color=colors, alpha=0.8)

ax.axvline(x=0, color='black', linewidth=2)
ax.set_xlabel('Score Change from Baseline', fontsize=12)
ax.set_title('Failed Experiments: What NOT to Do', fontsize=14, fontweight='bold')

# Add value labels
for bar, delta in zip(bars, deltas):
    x_pos = bar.get_width() - 0.5 if delta < 0 else bar.get_width() + 0.5
    ax.text(x_pos, bar.get_y() + bar.get_height()/2, f'{delta:+d}', 
            va='center', ha='right' if delta < 0 else 'left', 
            fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

### Key Lessons from Failures

| Lesson | Evidence | Recommendation |
|--------|----------|----------------|
| **Test one variable at a time** | V131 combined 6 changes, lost 6 points | Isolate each change |
| **Respect hardware constraints** | V135 maxed context, killed concurrency | Profile before scaling |
| **Simpler is often better** | V127 verbose prompts, V142 Bayesian voting | Start simple, add complexity only if proven |
| **Don't trust intuition** | Self-refinement, verification both failed | Measure everything |
| **Model-pipeline coupling matters** | Fine-tuned model failed with different pipeline | Use matched configurations |

## 8. Interactive Demonstrations

### Demo 1: Entropy-Weighted Voting in Action

In [ ]:
def generate_solution_attempt(answer: int, confidence: str) -> Dict:
    """
    Generate a simulated solution attempt with realistic entropy profile.
    
    Args:
        answer: The answer this attempt produces
        confidence: 'high', 'medium', or 'low'
    """
    n_tokens = 100
    
    if confidence == 'high':
        # Confident solution: low entropy at end
        probs = [[0.9 + np.random.uniform(0, 0.09), 0.01 + np.random.uniform(0, 0.05)] 
                 for _ in range(n_tokens)]
        # Normalize and make final tokens very confident
        for i in range(80, 100):
            probs[i] = [0.98, 0.02]
    elif confidence == 'medium':
        probs = [[0.7 + np.random.uniform(0, 0.2), 0.1 + np.random.uniform(0, 0.1)] 
                 for _ in range(n_tokens)]
    else:  # low
        probs = [[0.4 + np.random.uniform(0, 0.3), 0.3 + np.random.uniform(0, 0.2)] 
                 for _ in range(n_tokens)]
    
    # Normalize probabilities
    probs = [[p/sum(ps) for p in ps] for ps in probs]
    
    return {'answer': answer, 'logprobs': probs, 'confidence': confidence}


# Simulate 8 parallel attempts
np.random.seed(42)
simulated_results = [
    generate_solution_attempt(42, 'high'),     # Attempt 1: correct, confident
    generate_solution_attempt(42, 'high'),     # Attempt 2: correct, confident
    generate_solution_attempt(42, 'medium'),   # Attempt 3: correct, medium
    generate_solution_attempt(37, 'low'),      # Attempt 4: wrong, uncertain
    generate_solution_attempt(37, 'low'),      # Attempt 5: wrong, uncertain
    generate_solution_attempt(42, 'high'),     # Attempt 6: correct, confident
    generate_solution_attempt(99, 'low'),      # Attempt 7: wrong, uncertain
    generate_solution_attempt(42, 'medium'),   # Attempt 8: correct, medium
]

print("SIMULATED VOTING DEMONSTRATION")
print("=" * 70)
print("\n8 parallel solution attempts:")
print("-" * 70)

for i, result in enumerate(simulated_results, 1):
    entropy = compute_weighted_entropy(result['logprobs'])
    weight = 1.0 / max(entropy, 1e-9)
    print(f"Attempt {i}: Answer = {result['answer']:5d} | "
          f"Confidence = {result['confidence']:6s} | "
          f"Entropy = {entropy:.4f} | "
          f"Vote Weight = {weight:.2f}")

# Run voting
winner, total_weight, details = select_answer(simulated_results)

print("\n" + "=" * 70)
print("VOTING RESULTS:")
print("-" * 70)

# Show weights by answer
answer_weights = defaultdict(float)
answer_counts = defaultdict(int)
for result in simulated_results:
    entropy = compute_weighted_entropy(result['logprobs'])
    weight = 1.0 / max(entropy, 1e-9)
    answer_weights[result['answer']] += weight
    answer_counts[result['answer']] += 1

print(f"\n{'Answer':<10} {'Raw Votes':<12} {'Weighted Votes':<15} {'Winner?'}")
print("-" * 50)
for ans in sorted(answer_weights.keys()):
    is_winner = "✓ SELECTED" if ans == winner else ""
    print(f"{ans:<10} {answer_counts[ans]:<12} {answer_weights[ans]:<15.2f} {is_winner}")

print(f"\n{'='*70}")
print(f"FINAL ANSWER: {winner}")
print(f"\nNote: Answer 42 wins despite 37 having 2 votes because 42's votes")
print(f"come from HIGH CONFIDENCE solutions (low entropy → high weight).")

In [ ]:
# Visualize the voting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw votes
answers = list(answer_counts.keys())
raw_votes = [answer_counts[a] for a in answers]
weighted_votes = [answer_weights[a] for a in answers]

colors = ['#2ecc71' if a == 42 else '#e74c3c' for a in answers]

axes[0].bar([str(a) for a in answers], raw_votes, color=colors, alpha=0.8)
axes[0].set_title('Raw Vote Count\n(Simple Majority)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Answer')
axes[0].set_ylabel('Number of Votes')
for i, (a, v) in enumerate(zip(answers, raw_votes)):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontsize=12, fontweight='bold')

axes[1].bar([str(a) for a in answers], weighted_votes, color=colors, alpha=0.8)
axes[1].set_title('Entropy-Weighted Votes\n(Our Method)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Answer')
axes[1].set_ylabel('Weighted Vote Sum')
for i, (a, v) in enumerate(zip(answers, weighted_votes)):
    axes[1].text(i, v + 0.2, f'{v:.1f}', ha='center', fontsize=12, fontweight='bold')

# Highlight winner
axes[1].annotate('WINNER', xy=(0, weighted_votes[0]), xytext=(0.3, weighted_votes[0] + 2),
                 fontsize=12, fontweight='bold', color='green',
                 arrowprops=dict(arrowstyle='->', color='green'))

plt.suptitle('Why Entropy-Weighted Voting Outperforms Simple Majority', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Demo 2: Early Stopping Mechanism

In [ ]:
def simulate_early_stopping(attempts: int = 8, early_stop: int = 4):
    """
    Demonstrate the early stopping mechanism.
    
    Stop when `early_stop` attempts agree on the same answer.
    This saves compute time without sacrificing accuracy.
    """
    print(f"Early Stopping Simulation (stop at {early_stop} consensus)")
    print("=" * 60)
    
    np.random.seed(123)
    
    # Simulate answers (70% chance of correct answer 42)
    possible_answers = [42, 42, 42, 42, 42, 42, 42, 37, 99, 15]
    
    answer_counts = defaultdict(int)
    stopped_early = False
    stop_at = None
    
    for i in range(1, attempts + 1):
        answer = np.random.choice(possible_answers)
        answer_counts[answer] += 1
        
        # Check for consensus
        max_count = max(answer_counts.values())
        consensus_answer = max(answer_counts, key=answer_counts.get)
        
        status = ""
        if max_count >= early_stop and not stopped_early:
            stopped_early = True
            stop_at = i
            status = f" ← EARLY STOP! {early_stop} votes for {consensus_answer}"
        
        print(f"Attempt {i}: Answer = {answer:3d} | "
              f"Counts: {dict(answer_counts)} | "
              f"Max consensus: {max_count}{status}")
        
        if stopped_early:
            remaining = attempts - i
            print(f"\n→ Saved {remaining} attempts ({remaining/attempts*100:.0f}% compute)")
            break
    
    return stop_at, dict(answer_counts)

stop_at, counts = simulate_early_stopping()

## 9. Best Practices & Recommendations

### Configuration Checklist

Based on 157+ experiments, here's the optimal configuration:

```python
# BEST CONFIGURATION (V40 - 42/50)
config = {
    # Model
    'model': 'danielhanchen/gpt-oss-120b',
    'context_tokens': 65536,        # NOT 131K
    'kv_cache_dtype': 'fp8_e4m3',   # Essential for memory
    
    # Sampling
    'temperature': 1.0,             # NOT lower - need diversity
    'min_p': 0.02,
    
    # Voting
    'attempts': 8,
    'early_stop': 4,
    'voting': 'weighted_entropy',   # 5-component version
    
    # Prompts
    'prompt_style': 'simple',       # 3 lines only
}
```

### Do's and Don'ts

In [ ]:
recommendations = {
    "DO": [
        "Use simple 3-line prompts",
        "Set temperature to 1.0 with voting",
        "Use 65K context (not max)",
        "Implement position-weighted entropy",
        "Test one variable at a time",
        "Use early stopping at 4 consensus",
        "Profile memory before scaling context",
        "Measure everything - don't trust intuition",
    ],
    "DON'T": [
        "Use verbose multi-paragraph prompts (-9 pts)",
        "Set low temperature with voting (-8 pts)",
        "Max out context window (-6 pts)",
        "Use Bayesian/VOI voting (-11 pts)",
        "Combine multiple changes at once (-6 pts)",
        "Add self-refinement steps (-4 pts)",
        "Use mismatched fine-tuned models (-6 pts)",
        "Assume sophisticated = better",
    ]
}

print("RECOMMENDATIONS FROM 157+ EXPERIMENTS")
print("=" * 60)
print("\n✅ DO:")
for item in recommendations["DO"]:
    print(f"   • {item}")
print("\n❌ DON'T:")
for item in recommendations["DON'T"]:
    print(f"   • {item}")

## 10. Conclusion

### Summary of Contributions

1. **5-Component Weighted Entropy Voting** (+6 points over mean entropy)
   - Position weighting (40%) captures mathematical reasoning structure
   - Variance penalty catches inconsistent solutions
   - High-entropy penalty identifies guessing

2. **Simple Prompts Beat Complex Instructions** (+9 points)
   - 3-line prompts outperform detailed frameworks
   - Context conservation matters
   - Don't override model's learned strategies

3. **Comprehensive Ablation Studies**
   - Temperature 1.0 optimal for voting (diversity matters)
   - 65K context beats 131K (concurrency matters)
   - 8 attempts with early-stop at 4

4. **Failure Documentation**
   - Bayesian voting: -11 points
   - Self-refinement: -4 points
   - Combined changes: -6 points

### Gap Analysis

| Metric | Our Best | Competition Winner | Gap |
|--------|----------|-------------------|-----|
| Score | 42/50 | 46/50 | 4 points |

The 4-point gap may require:
- Different temperature + entropy weight combinations
- Tree-search methods (MCTS)
- Problem-specific prompting
- Ensemble approaches

### Links

- **GitHub:** https://github.com/PawanRamaMali/aimo3-inference-scaling
- **Kaggle Notebook:** https://www.kaggle.com/code/pawanmali/aimo3-gpt-oss-120b
- **Model (Kaggle):** https://www.kaggle.com/models/danielhanchen/gpt-oss-120b
- **Model (HuggingFace):** https://huggingface.co/unsloth/gpt-oss-120B

---

*This notebook documents findings from 157+ experiments conducted over 3 months. We hope this detailed analysis helps future practitioners avoid common pitfalls and build on our successes.*